# Hoja de Trabajo 2
Juan Diego Solís Martínez - 23720

Victor Manuel Pérez Chávez - 23714


## Task 1: Implementación completa
### Instalación y dependencias

In [1]:
!pip install torch torchvision matplotlib requests pillow --quiet

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import os
from PIL import Image
from io import BytesIO
import time

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

Usando dispositivo: cpu


### Parámetros fijos

In [3]:
# Task 1.1 - Parámetros fijos según la hoja de trabajo
Z_DIM = 100
IMG_SIZE = 64
IMG_CHANNELS = 3
FEATURES_G = 64
FEATURES_D = 64

# Task 1.2 - Parámetros de entrenamiento (DCGAN)
BATCH_SIZE = 32
NUM_EPOCHS = 50
LR = 2e-4
BETAS = (0.5, 0.999)

print('Hiperparámetros configurados correctamente.')

Hiperparámetros configurados correctamente.


### Dataset: Descarga de sprites desde el PokeAPI

In [4]:
def download_pokemon_sprites(save_dir='pokemon_sprites', max_pokemon=898):
    os.makedirs(save_dir, exist_ok=True)

    base_url = "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/"
    downloaded = 0
    failed = 0

    print(f'Descargando sprites de {max_pokemon} Pokémon:')

    for pokemon_id in range(1, max_pokemon + 1):
        filepath = os.path.join(save_dir, f'{pokemon_id}.png')

        if os.path.exists(filepath):
            downloaded += 1
            continue

        url = f"{base_url}{pokemon_id}.png"

        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content)).convert('RGBA')

                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[3])
                background.save(filepath)
                downloaded += 1

            else:
                failed += 1

        except Exception as e:
            failed += 1

        if pokemon_id % 100 == 0:
            print(f' - Progreso: {pokemon_id}/{max_pokemon} — descargados: {downloaded}, fallidos: {failed}')
        time.sleep(0.05)

    print(f'\nDescarga completada: {downloaded} sprites, {failed} fallidos.')
    return save_dir

sprite_dir = download_pokemon_sprites() # Descargar sprites

Descargando sprites de 898 Pokémon:
 - Progreso: 100/898 — descargados: 100, fallidos: 0
 - Progreso: 200/898 — descargados: 200, fallidos: 0
 - Progreso: 300/898 — descargados: 300, fallidos: 0
 - Progreso: 400/898 — descargados: 400, fallidos: 0
 - Progreso: 500/898 — descargados: 500, fallidos: 0
 - Progreso: 600/898 — descargados: 600, fallidos: 0
 - Progreso: 700/898 — descargados: 700, fallidos: 0
 - Progreso: 800/898 — descargados: 800, fallidos: 0

Descarga completada: 898 sprites, 0 fallidos.


In [5]:
class PokemonDataset(Dataset):
    # Dataset de sprites de Pokémon.
    # Aplica transformaciones: r000esize a 64x64, normalización a [-1, 1]
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(image_dir)
            if f.endswith('.png') or f.endswith('.jpg')
        ]
        print(f'Dataset: {len(self.image_files)} imágenes encontradas.')

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# Transformaciones: resize + tensor + normalización [-1, 1]
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

dataset = PokemonDataset(sprite_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

print(f'DataLoader: {len(dataloader)} batches de tamaño {BATCH_SIZE}')

Dataset: 898 imágenes encontradas.
DataLoader: 28 batches de tamaño 32
